<a href="https://colab.research.google.com/github/YugVarshney123/AI_Compute/blob/main/AI_Platforms_dep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Spark Environment Setup and Data Loading

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

spark=SparkSession.builder.appName("WaterAnalytics").getOrCreate()
sc=spark.sparkContext
df=spark.read.csv("DataSetAICompuetRelab.csv",header=True,inferSchema=True)


In [14]:
df.printSchema()
df.show(10,False)
print("Total Records:",df.count())

root
 |-- Record_ID: integer (nullable = true)
 |-- Supply_Date: date (nullable = true)
 |-- Zone: string (nullable = true)
 |-- Distribution_Station: string (nullable = true)
 |-- Water_Source: string (nullable = true)
 |-- Supply_Volume_Liters: double (nullable = true)
 |-- Consumption_Liters: double (nullable = true)
 |-- Water_Loss_Liters: double (nullable = true)
 |-- Pressure_PSI: double (nullable = true)
 |-- Pipeline_ID: string (nullable = true)
 |-- Leakage_Events: integer (nullable = true)
 |-- Water_Quality: string (nullable = true)
 |-- PH_Value: double (nullable = true)
 |-- Turbidity_NTU: double (nullable = true)
 |-- Consumer_Count: integer (nullable = true)
 |-- Maintenance_Cost: double (nullable = true)
 |-- System_Status: string (nullable = true)

+---------+-----------+-------+--------------------+--------------+--------------------+------------------+-----------------+------------+-----------+--------------+-------------+--------+-------------+--------------+-------

In [15]:
df.describe().show()

+-------+------------------+-------+--------------------+--------------+--------------------+------------------+------------------+------------------+-----------+------------------+-------------+------------------+-----------------+-----------------+------------------+-------------+
|summary|         Record_ID|   Zone|Distribution_Station|  Water_Source|Supply_Volume_Liters|Consumption_Liters| Water_Loss_Liters|      Pressure_PSI|Pipeline_ID|    Leakage_Events|Water_Quality|          PH_Value|    Turbidity_NTU|   Consumer_Count|  Maintenance_Cost|System_Status|
+-------+------------------+-------+--------------------+--------------+--------------------+------------------+------------------+------------------+-----------+------------------+-------------+------------------+-----------------+-----------------+------------------+-------------+
|  count|             10000|  10000|               10000|         10000|               10000|             10000|             10000|             1000

In [6]:
df.columns

['Record_ID',
 'Supply_Date',
 'Zone',
 'Distribution_Station',
 'Water_Source',
 'Supply_Volume_Liters',
 'Consumption_Liters',
 'Water_Loss_Liters',
 'Pressure_PSI',
 'Pipeline_ID',
 'Leakage_Events',
 'Water_Quality',
 'PH_Value',
 'Turbidity_NTU',
 'Consumer_Count',
 'Maintenance_Cost',
 'System_Status']

In [13]:
df.describe(['Distribution_Station']).show()
df.describe(['Consumption_Liters']).show()
df.describe(['Water_Loss_Liters']).show()
df.describe(['Supply_Volume_Liters']).show()
df.describe(['PH_Value']).show()
df.describe(['Turbidity_NTU']).show()

+-------+--------------------+
|summary|Distribution_Station|
+-------+--------------------+
|  count|               10000|
|   mean|                NULL|
| stddev|                NULL|
|    min|                 DS1|
|    max|                 DS9|
+-------+--------------------+

+-------+------------------+
|summary|Consumption_Liters|
+-------+------------------+
|  count|             10000|
|   mean| 86086.21900999984|
| stddev| 45789.96880588793|
|    min|           7206.98|
|    max|         193180.87|
+-------+------------------+

+-------+------------------+
|summary| Water_Loss_Liters|
+-------+------------------+
|  count|             10000|
|   mean|19623.204553999996|
| stddev|15324.337477240319|
|    min|            224.99|
|    max|          68736.19|
+-------+------------------+

+-------+--------------------+
|summary|Supply_Volume_Liters|
+-------+--------------------+
|  count|               10000|
|   mean|  105709.42356399984|
| stddev|   54595.56159590053|
|    min| 

Q2 RDD Programming and Transformation

In [16]:
rdd=sc.textFile("DataSetAICompuetRelab.csv")
header=rdd.first()


In [17]:
data=rdd.filter(lambda x:x!=header)
mapped=data.map(lambda x:x.split(","))
filtered=mapped.filter(lambda x: float(x[7])>1000)
flat=mapped.flatMap(lambda x:x)
print("Count:",data.count())
print("Collect:",mapped.take(5))


Count: 10000
Collect: [['1', '2026-01-25', 'Central', 'DS27', 'TreatmentPlant', '72854.04', '63503.1', '9350.94', '59.3', 'P480', '4', 'Excellent', '7.52', '5.49', '3240', '91535.33', 'Normal'], ['2', '2026-03-29', 'North', 'DS11', 'TreatmentPlant', '10989.33', '9112.23', '1877.1', '34.9', 'P143', '0', 'Excellent', '6.52', '4.63', '4896', '31194.67', 'Normal'], ['3', '2026-02-03', 'North', 'DS11', 'River', '163173.87', '150019.45', '13154.42', '72.6', 'P173', '5', 'Poor', '7.57', '5.36', '1354', '23166.08', 'Normal'], ['4', '2026-01-19', 'North', 'DS12', 'Groundwater', '130479.58', '113023.42', '17456.16', '58.6', 'P295', '2', 'Excellent', '8.11', '3.41', '1144', '53828.2', 'Maintenance'], ['5', '2026-01-25', 'Central', 'DS10', 'Groundwater', '87431.83', '66273.91', '21157.92', '25.3', 'P298', '1', 'Excellent', '7.13', '3.02', '922', '89765.91', 'Normal']]


In [20]:
from functools import reduce
nums=mapped.map(lambda x: float(x[5]))
print("Total Supply:",nums.reduce(lambda a,b:a+b))
zone_supply=mapped.map(lambda x:(x[2],float(x[5]))).reduceByKey(lambda a,b:a+b)
zone_supply.collect()

Total Supply: 1057094235.6399982


[('Central', 210179145.88000005),
 ('North', 214628262.2100001),
 ('West', 214516964.19999987),
 ('South', 206422486.65000004),
 ('East', 211347376.69999984)]

Q3. Spark DataFrame Operations

In [21]:
df.select("Zone","Distribution_Station","Supply_Volume_Liters","Consumption_Liters").show()


+-------+--------------------+--------------------+------------------+
|   Zone|Distribution_Station|Supply_Volume_Liters|Consumption_Liters|
+-------+--------------------+--------------------+------------------+
|Central|                DS27|            72854.04|           63503.1|
|  North|                DS11|            10989.33|           9112.23|
|  North|                DS11|           163173.87|         150019.45|
|  North|                DS12|           130479.58|         113023.42|
|Central|                DS10|            87431.83|          66273.91|
|  North|                 DS1|           152778.64|         120693.18|
|  North|                DS19|           157997.93|         114399.66|
|  South|                DS14|            69118.79|          59602.42|
|  South|                DS10|           139833.28|         103052.95|
|   West|                DS25|           157212.05|         139306.36|
|  North|                 DS7|            115749.7|         113319.37|
|   Ea

In [22]:
df.filter((col("Water_Loss_Liters")>1000)&(col("Leakage_Events")>2)).show()


+---------+-----------+-------+--------------------+--------------+--------------------+------------------+-----------------+------------+-----------+--------------+-------------+--------+-------------+--------------+----------------+-------------+
|Record_ID|Supply_Date|   Zone|Distribution_Station|  Water_Source|Supply_Volume_Liters|Consumption_Liters|Water_Loss_Liters|Pressure_PSI|Pipeline_ID|Leakage_Events|Water_Quality|PH_Value|Turbidity_NTU|Consumer_Count|Maintenance_Cost|System_Status|
+---------+-----------+-------+--------------------+--------------+--------------------+------------------+-----------------+------------+-----------+--------------+-------------+--------+-------------+--------------+----------------+-------------+
|        1| 2026-01-25|Central|                DS27|TreatmentPlant|            72854.04|           63503.1|          9350.94|        59.3|       P480|             4|    Excellent|    7.52|         5.49|          3240|        91535.33|       Normal|
|   

In [23]:
df.orderBy(col("Water_Loss_Liters").desc()).show()

+---------+-----------+-------+--------------------+--------------+--------------------+------------------+-----------------+------------+-----------+--------------+-------------+--------+-------------+--------------+----------------+-------------+
|Record_ID|Supply_Date|   Zone|Distribution_Station|  Water_Source|Supply_Volume_Liters|Consumption_Liters|Water_Loss_Liters|Pressure_PSI|Pipeline_ID|Leakage_Events|Water_Quality|PH_Value|Turbidity_NTU|Consumer_Count|Maintenance_Cost|System_Status|
+---------+-----------+-------+--------------------+--------------+--------------------+------------------+-----------------+------------+-----------+--------------+-------------+--------+-------------+--------------+----------------+-------------+
|     5050| 2026-01-29|  North|                 DS9|         River|           199219.82|         130483.63|         68736.19|        23.5|       P731|             1|    Excellent|    8.31|         9.85|          1584|        65171.69|       Normal|
|   

In [24]:

df.groupBy("Zone").agg(
sum("Supply_Volume_Liters").alias("TotalSupply"),
sum("Consumption_Liters").alias("TotalConsumption"),
sum("Water_Loss_Liters").alias("TotalLoss"),
avg("Pressure_PSI").alias("AvgPressure"),
avg("Maintenance_Cost").alias("AvgMaintenance")
).show()

+-------+--------------------+--------------------+-------------------+------------------+------------------+
|   Zone|         TotalSupply|    TotalConsumption|          TotalLoss|       AvgPressure|    AvgMaintenance|
+-------+--------------------+--------------------+-------------------+------------------+------------------+
|  South|2.0642248665000013E8| 1.668846068800003E8|3.953787977000007E7|50.380253164556926| 52108.16070886062|
|Central|2.1017914588000005E8| 1.712170387399998E8|3.896210713999991E7| 50.11089258698945| 51126.89768028241|
|   East|2.1134737670000002E8|      1.7249534199E8|3.885203470999998E7| 50.15527763881939| 50354.88592796408|
|   West|2.1451696420000035E8|1.7504741414000028E8|3.946955005999999E7|50.432172633643944|49659.052594409026|
|  North|2.1462826221000025E8|1.7521778835000008E8|3.941047385999994E7|49.229690618762454|50557.914351297404|
+-------+--------------------+--------------------+-------------------+------------------+------------------+



Q 4Exploratory Data Analysis using Spark SQL

In [26]:
df.createOrReplaceTempView("water")

In [27]:
spark.sql("select Zone,sum(Consumption_Liters) c from water group by Zone order by c desc limit 5").show()

+-------+--------------------+
|   Zone|                   c|
+-------+--------------------+
|  North|1.7521778835000008E8|
|   West|1.7504741414000028E8|
|   East|      1.7249534199E8|
|Central| 1.712170387399998E8|
|  South| 1.668846068800003E8|
+-------+--------------------+



In [28]:
spark.sql("select Zone,sum(Water_Loss_Liters) TotalLoss,avg(Water_Loss_Liters) AvgLoss from water group by Zone").show()

+-------+-------------------+------------------+
|   Zone|          TotalLoss|           AvgLoss|
+-------+-------------------+------------------+
|  South|3.953787977000007E7|20019.179630379782|
|Central|3.896210713999991E7|19648.062097831524|
|   East|3.885203470999998E7|19435.735222611296|
|   West|3.946955005999999E7|19357.307533104457|
|  North|3.941047385999994E7| 19665.90511976045|
+-------+-------------------+------------------+



In [30]:
spark.sql("select Distribution_Station, round(sum(Consumption_Liters)/sum(Supply_Volume_Liters)*100,2) Efficiency from water group by Distribution_Station").show()

+--------------------+----------+
|Distribution_Station|Efficiency|
+--------------------+----------+
|                 DS1|     80.57|
|                DS29|     80.96|
|                DS28|     81.54|
|                DS27|     81.03|
|                 DS9|     82.09|
|                 DS3|     80.67|
|                DS21|     82.72|
|                DS13|     82.25|
|                DS17|     80.94|
|                 DS4|     81.61|
|                DS30|     81.05|
|                 DS8|     79.98|
|                 DS5|     81.59|
|                DS18|     82.75|
|                DS26|     82.19|
|                DS16|     81.49|
|                DS22|     80.67|
|                DS10|     80.94|
|                DS24|      80.4|
|                DS19|     80.98|
+--------------------+----------+
only showing top 20 rows


In [31]:
spark.sql("select month(Supply_Date) Month,sum(Supply_Volume_Liters) Supply from water group by month(Supply_Date) order by Month").show()

+-----+--------------------+
|Month|              Supply|
+-----+--------------------+
|    1| 1.759291433300001E8|
|    2|1.5768824718000016E8|
|    3|1.8448445542999986E8|
|    4| 1.810309662200003E8|
|    5|1.7895177555000037E8|
|    6|1.7900964792999995E8|
+-----+--------------------+



In [32]:
spark.sql("select month(Supply_Date) Month,sum(Consumption_Liters) Consumption from water group by month(Supply_Date) order by Month").show()

+-----+--------------------+
|Month|         Consumption|
+-----+--------------------+
|    1| 1.435833950699999E8|
|    2|1.2768821915000013E8|
|    3|1.4976594985999998E8|
|    4|1.4759754209999993E8|
|    5| 1.461438116700001E8|
|    6|1.4608327225000006E8|
+-----+--------------------+



In [33]:
spark.sql("select Zone,sum(Leakage_Events) Leakages from water group by Zone order by Leakages desc").show()

+-------+--------+
|   Zone|Leakages|
+-------+--------+
|   West|    5036|
|  South|    4999|
|  North|    4975|
|   East|    4964|
|Central|    4880|
+-------+--------+



In [34]:
spark.sql("select Water_Source,avg(PH_Value) AvgPH,avg(Turbidity_NTU) AvgTurbidity from water group by Water_Source").show()

+--------------+-----------------+------------------+
|  Water_Source|            AvgPH|      AvgTurbidity|
+--------------+-----------------+------------------+
|     Reservoir| 7.50918622848202| 5.155571205007824|
|TreatmentPlant|7.509080971659923| 5.049720647773277|
|   Groundwater|7.488087234893963|4.9499679871948805|
|         River|7.511353535353514| 5.052343434343431|
+--------------+-----------------+------------------+



Q5. ETL Pipeline Development (4 Marks)

In [38]:
df=spark.read.csv("DataSetAICompuetRelab.csv",header=True,inferSchema=True)

In [35]:
etl=df.dropDuplicates().na.drop()


In [36]:
etl=etl.withColumn("Supply_Date",to_date("Supply_Date"))

In [37]:
etl.write.mode("overwrite").parquet("Processed_Water_Data")
print("Saved as Parquet")

Saved as Parquet


ETL Workflow

CSV -> Extract -> Remove Duplicates -> Handle Missing Values -> Type Conversion -> Parquet

Q6. Machine Learning using Spark MLlib


In [39]:
features=["Supply_Volume_Liters","Water_Loss_Liters","Pressure_PSI","Consumer_Count","Leakage_Events"]



In [40]:
assembler=VectorAssembler(inputCols=features,outputCol="features")


In [41]:
ml=assembler.transform(df).select("features","Consumption_Liters")
train,test=ml.randomSplit([0.8,0.2],42)

In [42]:
lr=LinearRegression(featuresCol="features",labelCol="Consumption_Liters")
model=lr.fit(train)

In [43]:
pred=model.transform(test)
pred.select("prediction","Consumption_Liters").show()
from pyspark.ml.evaluation import RegressionEvaluator
ev=RegressionEvaluator(labelCol="Consumption_Liters",predictionCol="prediction",metricName="rmse")
print("RMSE:",ev.evaluate(pred))

+------------------+------------------+
|        prediction|Consumption_Liters|
+------------------+------------------+
| 8021.990000002444|           8021.99|
| 9230.229999999923|           9230.23|
| 8157.680000000634|           8157.68|
| 7651.090000002074|           7651.09|
| 8330.649999999407|           8330.65|
|10188.049999999623|          10188.05|
|  9107.76999999997|           9107.77|
|  9711.19000000158|           9711.19|
|10863.620000000976|          10863.62|
| 8993.480000001484|           8993.48|
| 9398.950000001681|           9398.95|
|10434.300000000034|           10434.3|
| 9754.460000001756|           9754.46|
| 9535.670000001288|           9535.67|
|  9994.48000000222|           9994.48|
| 8740.470000001123|           8740.47|
| 9094.620000000266|           9094.62|
| 10319.50000000158|           10319.5|
|11331.780000001532|          11331.78|
| 9862.890000001978|           9862.89|
+------------------+------------------+
only showing top 20 rows
RMSE: 1.1896148

In [44]:
from pyspark.ml.evaluation import RegressionEvaluator
ev=RegressionEvaluator(labelCol="Consumption_Liters",predictionCol="prediction",metricName="rmse")
print("RMSE:",ev.evaluate(pred))

RMSE: 1.1896148809052743e-09


7. Deployment using DevOps Tool

B.
1.Code Push:
The developer writes code and pushes it to a GitHub repository.
2.CI Pipeline Trigger
GitHub Actions, Jenkins, or GitLab CI automatically detects the new commit and starts the pipeline.
3.Checkout Source Code
The pipeline downloads the latest project code from the repository.

4.Build the Project
The application is prepared for execution by compiling or packaging it if necessary.
5.Run Automated Tests
6.Unit tests and integration tests are executed to verify that the application works correctly.
7.Deploy Application
If all tests pass, the application is automatically deployed to a production or cloud environment.
